## 1. Async / Coroutines, Event Loop
- A coroutine is a function that can suspend itself at well-defined points and resume later, preserving its local state across suspensions.
- A coroutine object is produced by using `async def`.
    - every coroutine must eventually be driven by something. If none, it silently does nothing.
    - 3 ways to drive it: `await <>` inside another coroutine (most common), `asyncio.run(<>)` as top-entry point, or `asyncio.create_task(<>)`
- The coroutine decides when to yield control — at every await.

- An event loop is a single-threaded loop that:
    - Maintains a queue of ready-to-run coroutines
    - Maintains a set of I/O watchers (socket descriptors waiting for data)
    - Each iteration: run everything ready, then ask the OS "which sockets have data?" (select/epoll), then wake up the waiting coroutines
    - Example: Event Loop (single thread)
        ```txt
        ┌─────────────────────────────────────────────────────┐
        │                                                     │
        │   ready queue: [coro_A, coro_C]                     │
        │                                                     │
        │   1. Run coro_A until it hits `await`  ──► suspends │
        │   2. Run coro_C until it hits `await`  ──► suspends │
        │   3. Ask OS: any I/O ready?                         │
        │      OS: socket_42 has data                         │
        │   4. Wake coro_B (was waiting on socket_42)         │
        │   5. Add coro_B to ready queue                      │
        │   6. Loop                                           │
        │                                                     │
        └─────────────────────────────────────────────────────┘
        ``
- An event loop is created by `asyncio.run()`:

    ```txt
    asyncio.run(coro)
    │
    ├── 1. Creates a new event loop
    ├── 2. Runs the coroutine on that loop until completion
    └── 3. Closes and cleans up the loop
    ```

- Common patterns:
    - You can only call it once at the top level (not inside an already-running loop — the Jupyter error, which needs you to join the current loop).
    - ou typically have one `asyncio.run()` per program, as the entry point. Everything else inside uses await or create_task, which reuses the already-running loop created by asyncio.run()

- **WARNING**: If a coroutine calls a synchronous blocking function (e.g., `time.sleep()`, a blocking DB call, requests.get()), the entire event loop freezes
- Note on Jupyter Notebook:
    - Jupyter itself runs an event loop permanently in the background — it's how it handles kernel communication, cell execution, and UI updates.
    - When you call `asyncio.run()`, it tries to create and start a new event loop, but the rule is: one event loop per thread. hence it raises RuntimeError: can't nest event loops
        ```txt
        Jupyter kernel thread
        └── Event loop (already running) ◄── Jupyter owns this
                └── Your cell calls asyncio.run()
                        └── RuntimeError: can't nest event loops
        ```
    - hence instead of using `asyncio.run()` and creates a new event loop, use `await` directly to join the current running event loop.

In [2]:
import asyncio

 
async def coro_A():
    print("A: start")
    await asyncio.sleep(1)
    print("A: resume")

async def coro_B():
    print("B: start")
    await asyncio.sleep(0.5)
    print("B: resume")

# A starts first as it appears first,
# A then sleeps for 1s
# During this 1s, the thread is freed up, so B starts
# B sleeps for 0.5s and resumes, earlier than when A resumes
# A finally resumes

# in python script:
# asyncio.run(asyncio.gather(coro_A(), coro_B()))

# in Jupyter notebook with existing event loop, use await to join that loop
await asyncio.gather(coro_A(), coro_B())

# option 2: patch with nest_asyncio
# import nest_asyncio
# nest_asyncio.apply()   # For Jupyter notebook only: patches asyncio to allow nested event loops
# asyncio.run(asyncio.gather(coro_A(), coro_B()))

A: start
B: start
B: resume
A: resume


[None, None]

## 2. Working with yield - Regular Generator vs Async Generator
[Reference](https://levelup.gitconnected.com/7-python-async-generator-things-i-wish-i-knew-earlier-c769a32d4413)

### Definitions
- Regular generator: 
    - just a normal `def` function that uses `yield`.
    - produces values lazily when invoked only.
- Async generator:
    - a coroutine `async def` function that uses `yield`.
    - produces values lazily when invoked only
### Invocation
- Regular generator: use `next()` or `for` loop.
- Async generator: use `await anext()` or `async for` loop.
### Blocking behavior
- Regular generators cannot use `await`, and blocks the event loop.
- Async generators can use `await` and free up the event loop.

An example to demonstrate the use of async generator - handling concurrent incoming requests to the generator.
- `token_stream()`: Each instance of the normal sync generator will block the thread - as seen from the `handle_client()` function output, each client call generates the full message before the next client can call.
- `atoken_stream()`: Each instance of the async generator does not block the thread, and another instance can start running. As seen from the `ahandle_client()` outputs, all 3 client calls can execute in parallel without waiting one to complete. Overall, the total execution time is much lower. This cuts the total execution time dramatically.

In [8]:
def token_stream():
    for token in ["Hello", " world", "!"]:
        time.sleep(0.2)
        yield token

def handle_client(client_id:int):
    gen = token_stream()
    for token in gen:
        print(f"Client {client_id}: {token}", end="\t")

tik = time.time()
[handle_client(i) for i in range(3)]
tok = time.time()
duration = tok - tik
print()
print(f"Duration: {round(duration,3)} seconds")

Client 0: Hello	Client 0:  world	Client 0: !	Client 1: Hello	Client 1:  world	Client 1: !	Client 2: Hello	Client 2:  world	Client 2: !	
Duration: 1.803


In [9]:
async def atoken_stream():
    for token in ["Hello", " world", "!"]:
        await asyncio.sleep(0.2)
        yield token

async def ahandle_client(client_id: int):
    agen = atoken_stream()
    async for token in agen:
        print(f"Client {client_id}: {token}", end="\t")

tik = time.time()
await asyncio.gather(*[ahandle_client(i) for i in range(3)])
tok = time.time()
duration = tok - tik
print()
print(f"Duration: {round(duration,3)} seconds")

Client 0: Hello	Client 1: Hello	Client 2: Hello	Client 0:  world	Client 1:  world	Client 2:  world	Client 0: !	Client 1: !	Client 2: !	
Duration: 0.602 seconds


## WARNING
- When async is not written properly, it may contain blocking codes inside, this blocks the thread when executing an instance of the coroutine async generator and defeats the purpose of having it written in async.
- In the example below, `time.sleep()` is a blocking code.

In [ ]:
import time


async def coro_bad():
    print("bad: start")
    time.sleep(1)           # ← blocks the event loop, B cannot start
    print("bad: resume")

async def coro_B():
    print("B: start")
    await asyncio.sleep(0.5)
    print("B: resume")

await asyncio.gather(coro_bad(), coro_B())

bad: start
bad: resume
B: start
B: resume


[None, None]

## 3. LLM Streaming
LLM streaming is fundamentally a waiting problem, not a computing problem.

When you call a streaming LLM API:
- You send a request, then wait for tokens to arrive over a network socket
- Tokens arrive incrementally, with gaps between them
- Each gap is pure I/O wait — your CPU is doing nothing useful

Without async, each gap blocks that thread. 
- Other threads are free and can run (the GIL is released during I/O wait). 
- But you still need 100 threads for 100 users: expensive in memory and OS scheduling overhead, most of them just sitting idle waiting for the next token.

    ```txt
    Thread calls API ──► waits ──► token ──► waits ──► token ──► waits ──► done
                        (blocked)           (blocked)           (blocked)
    ```

With async, the event loop lets a coroutine call the API and suspend itself at await. 
- The event loop uses that waiting time to run other coroutines, resuming this one when the next token arrives.

    ```txt
    Coroutine calls API ──► suspends at await ──► token arrives ──► resumes ──► yields token
                            (event loop runs      (OS notifies
                            other coroutines)     event loop)
    ```

### Extra: Thread pool
- In production, instead of 1 thread for all users, there is a fixed pool of worker threads, each thread runs its own event loop, handling many coroutines concurrently. 
- This is what production servers like uvicorn do — you configure worker count to match CPU cores, each worker is a thread (or process) with its own event loop.
- The number of threads is typically set to number of CPU cores — beyond that you get diminishing returns since threads compete for cores rather than adding capacity.
    ```txt
    Thread Pool (e.g. 4 threads)
    ┌─────────────────────────────────────────┐
    │ Thread 1 → Event Loop → User 1,5,9...  │
    │ Thread 2 → Event Loop → User 2,6,10... │
    │ Thread 3 → Event Loop → User 3,7,11... │
    │ Thread 4 → Event Loop → User 4,8,12... │
    └─────────────────────────────────────────┘
    ```

In [ ]:
import asyncio


# simulating a streaming LLM service:
# equivalent: stream=True for OpenAI vanilla chat completion client class
async def mock_llm_stream():
    tokens = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]
    for token in tokens:
        await asyncio.sleep(0.2) # simulates network gaps
        yield token


async def handle_user(user_id: int):
    async for token in mock_llm_stream():
        print(f"User {user_id}: {token}", end="\t")
        # print("\n")

await asyncio.gather(*[handle_user(i) for i in range(3)])

User 0: The	User 1: The	User 2: The	User 0: quick	User 1: quick	User 2: quick	User 0: brown	User 1: brown	User 2: brown	User 0: fox	User 1: fox	User 2: fox	User 0: jumps	User 1: jumps	User 2: jumps	User 0: over	User 1: over	User 2: over	User 0: the	User 1: the	User 2: the	User 0: lazy	User 1: lazy	User 2: lazy	User 0: dog	User 1: dog	User 2: dog	

[None, None, None]

## 2. OpenAI Streaming and Handling
Some notes on the OpenAI response schema:
1. `if not chunk.choices: continue`
    - Protocol guard. The final [DONE] chunk has choices=[] — skip it entirely before indexing.
2. `token = chunk.choices[0].delta.content`
    - Parses the actial text fragment from the chunk data structure.
    - `delta` is the actual incremental token (a word or partial word) to return to the caller - depends on the underlying tokenizer.
    - can be None on the first chunk (role assignment) and last real chunk (stop signal)
3. `if token --- yield token`
    - returns the actual word/partial word token to caller - the `handle_user()` coroutine in this case, filter off the None cases.
    - suspends the whole coroutine.
    - the local state of the coroutine is preserved until the next iteration is called by `__anext()__`

The OpenAI `ChatCompletionChunk` schema. See [OpenAI documentation](https://developers.openai.com/api/reference/go/resources/chat/subresources/completions) for more info.
```txt
ChatCompletionChunk
├── id                          # same across all chunks in one response
├── model                       # e.g. "gpt-4o"
├── created                     # unix timestamp
├── choices: [
│     └── Choice
│           ├── index           # 0 for single completion
│           ├── finish_reason   # None mid-stream, "stop" on last real chunk
│           └── delta
│                 ├── role      # "assistant" on first chunk only, None after
│                 └── content   # the token string, None on first/last chunks
│   ]
└── usage                       # None mid-stream, populated on final chunk*

```

In [10]:
import asyncio
from openai import AsyncAzureOpenAI, AzureOpenAI
from dotenv import load_dotenv
import os

load_dotenv()


deployment_name = "gpt-4o"
endpoint = "https://local-rag-resource.services.ai.azure.com/"

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-01"
)

# smoke test the connection:
response = client.chat.completions.create(
    model=deployment_name,
    messages=[{"role": "user", "content": "hi there"}],
    stream=False
)
response.choices[0].message.content

'Hello! 😊 How can I assist you today?'

### Method 1: Normal streaming (synchronous) - similar to a normal generator
- As the argument `stream=True` returns a generator-like object, we need to iterate it to get the tokens one by one.
- `stream` is a streaming response object, not the full completion.
- As the tokens per minute rate is quite fast, get the LLM to generate something longer so that you can observe the streaming.

In [ ]:
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-01"
)

stream = client.chat.completions.create(
    model=deployment_name,
    messages=[{"role": "user", "content": "hi there. read me a paragraph from a famous book."}],
    stream=True,
)

for chunk in stream:
    if not chunk.choices:
        continue
    token = chunk.choices[0].delta.content
    if token:
        print(token, end="", flush=True) # must use flush to see streaming effect

Of course! Here's a passage from *Pride and Prejudice* by Jane Austen, published in 1813:

*"It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife. However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered as the rightful property of some one or other of their daughters."*

Let me know if you'd like me to read from a different book!

- Instead of printing out the tokens as above, we create a wrapper generator to yield the tokens to another client handler, which helps demonstrate the difference in speed and concurrency of serving users.
- If you use the normal AzureOpenAI client, a synchronous one, to serve multiple users, it will create the blocking issue. Each subsequent user needs to wait for whoever before him/her to finish the client call.

In [ ]:
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-01"
)


def llm_stream_sync(prompt: str):
    stream = client.chat.completions.create(
        model=deployment_name,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )

    for chunk in stream:
        if not chunk.choices:
            continue
        token = chunk.choices[0].delta.content
        if token:
            yield token

def handle_user_sync(user_id: int):
    for token in llm_stream_sync("hi"):
        print(f"User {user_id}: {token}", end="\t", flush=True) 


tik = time.time()
result = [handle_user_sync(i) for i in range(3)]
tok = time.time()
duration = tok - tik
print()
print(f"Duration: {round(duration, 3)} seconds.")

User 0: Hello	User 0: !	User 0:  How	User 0:  can	User 0:  I	User 0:  assist	User 0:  you	User 0:  today	User 0: ?	User 0:  😊	User 1: Hello	User 1: !	User 1:  How	User 1:  can	User 1:  I	User 1:  assist	User 1:  you	User 1:  today	User 1: ?	User 1:  😊	User 2: Hello	User 2: !	User 2:  How	User 2:  can	User 2:  I	User 2:  assist	User 2:  you	User 2:  today	User 2: ?	User 2:  😊	
Duration: 4.635 seconds.


### Method 2: Async Streaming

Token generation and handling - non-blocking when waiting for LLM generating token
1. `stream = await client.chat.completions.create()`
    - As the `AsyncAzureOpenAI` object is an async generator, it must be awaited.
    - `await` is the first suspension point of the wrapper, waits for the API service to open the stream, hands control back to the event loop for other tasks such as handling other requests.
3. `async for chunk in stream`
    - As the wrapper is an async generator, we need to iterate it (with async) to start receiving tokens and yield it to the client side.
    - Each iteration calls `__anext__()` on the response object, suspends the coroutine when waiting for the next chunk to arrive over the network, resumes when a chunk arrives, and repeats.

Caller side - non-blocking when waiting for the llm_stream() on 1 user.
- Use `asyncio.gather()` to schedule the coroutines inside the bracket.
- Use `*[handle_user(i) for i in range(...)]` to unpack the list to individual items, equivalent to `gather(handle_user(0), handle_user(1),...)`
- All `handle_user()` coroutines run concurrently on a single thread.
- Whenever one suspends, the event loop resumes some others.
- Extremely scalable: each coroutine comsumes a few KB.

In [ ]:
client = AsyncAzureOpenAI(
    azure_endpoint=endpoint,
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-01"
)

async def llm_stream(prompt: str):
    stream = await client.chat.completions.create(
        model=deployment_name,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )
    async for chunk in stream:
        # the final [DONE] chunk has empty choices list, 
        if not chunk.choices:
            continue # skip to the next chunk
        token = chunk.choices[0].delta.content
        if token:
            yield token


async def handle_user(user_id: int):
    async for token in llm_stream("hi"):
        print(f"User {user_id}: {token}", end="\t", flush=True)

# "server" function - schedule the coroutines
# this takes 2.2s
tik = time.time()
await asyncio.gather(*[handle_user(i) for i in range(3)])
tok = time.time()
duration = tok - tik
print()
print(f"Duration: {round(duration, 3)} seconds.")

User 2: Hello	User 2: !	User 2:  How	User 2:  can	User 2:  I	User 2:  assist	User 2:  you	User 2:  today	User 2: ?	User 2:  😊	User 0: Hello	User 0: !	User 0:  How	User 0:  can	User 0:  I	User 0:  assist	User 0:  you	User 0:  today	User 0: ?	User 0:  😊	User 1: Hello	User 1: !	User 1:  How	User 1:  can	User 1:  I	User 1:  assist	User 1:  you	User 1:  today	User 1: ?	User 1:  😊	
Duration: 1.809 seconds.


### Patterns - good practices
1. Await once to initialize, then stream. `await` establishes the connection, `async for` consumes it. These are two distinct operations — don't conflate them.
    ```python
    response = await client.chat.completions.create(..., streaam=True)
    async for chunk in response:
        yield chunk
    ```
2. Async generator as stream abstraction: abstrating the pattern 1 above in another coroutine to pass arguments and call by something else.
    ```python
    # Abstration:
    async def llm_stream(prompt: str):
        response = await client.chat.completions.create(..., streaam=True) # uses `prompt` arg
        async for chunk in response:
            # process chunk
            # produce token
            yield token

    # Caller:
    async for token in llm_stream(prompt=...):
        print(token)
    ```

3. Chaining: If the caller is adding any further logic, extend it
    ```python
    async def process_stream(prompt: str):
        async for token in llm_stream(prompt):
            token = token.strip()
            if token:
                yield token
    ```

4. Guard the protocol boundaries explicitly
    - streaming protocols may have structural edge cases at the start or end, with different schemas.
    - handle them explicitly
    ```python
    async for chunk in response:
        if not chunk.choices:       # if it's [DONE] chunk, there is empty choices []
            continue
        token = chunk.choices[0].delta.content
        if token:                   # yield to caller if the delta content is not empty (None)
            yield token
    ```

5. Use a single shared async client
    - Use a shared HTTP client initialized at the module level for all requests
    - This is to maintain connection pool and avoid recreating TCP handshake overhead on every call.
    ```python
    client = AsyncAzureOpenAI(...)   # module level — created once
    async def llm_stream(prompt):
        response = await client.chat.completions.create(...)  # reuse client
    ```

### Antipatterns
1. Use a blocking code inside `async def`
    - Use the async equivalent of `httpx.AsyncClient` or the official async SDK client.
    ```python
    # bad
    async def llm_stream(prompt):
        response = requests.post(url, ...)    # blocks the event loop entirely
        yield response.text
    ```

2. Collect the full response before yielding to caller inside `async def`
    - Buffers everything in memory and delivers nothing until the stream ends. The entire point of streaming is discarded.
    ```python
    # bad
    async def llm_stream(prompt):
        tokens = []
        async for chunk in response:
            tokens.append(chunk)
        for token in tokens:          # defeats streaming entirely
            yield token
    ```

3. Confusing when to use `await`, when to use `async for`:
    - `async def` functions behave differently depending on whether they contain yield
    - 